# Sales Prediction Using Python

## 1. Project Overview

**Objective:** Predict product sales from advertising expenditure on TV, radio, and newspaper.

This is a **supervised regression** problem because the target (`sales`) is a continuous numerical value.

### Dataset columns
- `TV`: advertising expenditure on TV
- `radio`: advertising expenditure on radio
- `newspaper`: advertising expenditure on newspaper
- `sales`: target variable to predict
- `Unnamed: 0`: index-like identifier; not used as a predictor

### Workflow
1. Load and inspect the data
2. Perform exploratory data analysis (EDA)
3. Prepare features and target
4. Split into training and testing sets
5. Train and compare regression models
6. Tune a Decision Tree
7. Compare a Random Forest model
8. Select, evaluate, and save the final pipeline
9. Load the saved model and predict sales for new data

**Important:** Run this notebook from the project folder that contains `Advertising (1).csv`. The notebook saves `sales_prediction_pipeline.pkl` in the current working directory.

## 2. Import Libraries

Import the libraries used throughout the notebook in one place.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 3. Load the Dataset

Read the CSV file into a Pandas DataFrame. If Python cannot find the file, check the current working directory and make sure the CSV filename matches.

In [ ]:
print("Current working directory:", os.getcwd())
print("Files in current directory:", os.listdir())

df = pd.read_csv("Advertising (1).csv")
df.head()

## 4. Inspect the Dataset

Check the number of rows and columns, data types, summary statistics, missing values, and duplicate rows.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
display(df.isnull().sum())

print("Number of duplicate rows:", df.duplicated().sum())

### Handle duplicates and the index-like column

The original dataset has an `Unnamed: 0` column that acts like an identifier, not an advertising feature. We exclude it from modelling. Duplicate rows are checked before removing them; this code removes exact duplicate rows if any exist.

In [ ]:
df = df.drop_duplicates().copy()
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

print("Shape after cleanup:", df.shape)
print("Remaining columns:", df.columns.tolist())
df.head()

## 5. Exploratory Data Analysis (EDA)

EDA helps us understand the distributions of the variables and their relationships. Correlation describes association; it does not prove that advertising expenditure causes sales to change.

In [ ]:
correlation_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplots show how each advertising channel relates to sales.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feature in zip(axes, ["TV", "radio", "newspaper"]):
    sns.scatterplot(data=df, x=feature, y="sales", ax=ax)
    ax.set_title(f"{feature} vs Sales")

plt.tight_layout()
plt.show()

### EDA notes

In the recorded analysis, TV advertising had the strongest positive correlation with sales (approximately 0.78 in the correlation matrix after excluding the identifier), followed by radio (approximately 0.58). Newspaper had a weaker positive correlation (approximately 0.23). These correlations are descriptive and do not establish causation.

## 6. Select Features and Target

`X` contains the three input features. `y` contains the sales value the model learns to predict.

In [ ]:
X = df[["TV", "radio", "newspaper"]]
y = df["sales"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.head())

## 7. Split into Training and Testing Sets

Use 80% of the data for training and 20% for testing. `random_state=42` makes the split reproducible. The test set is held aside to estimate performance on unseen rows.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 8. Helper Function for Evaluation

Use the same metrics for each model so the comparison is fair.

- **MAE:** average absolute prediction error; lower is better.
- **MSE:** average squared error; lower is better.
- **RMSE:** square root of MSE, expressed in sales units; lower is better.
- **R²:** proportion of target variation explained by the model on the evaluated data; higher is generally better.

In [ ]:
def evaluate_regression(model, X_eval, y_eval):
    predictions = model.predict(X_eval)
    return {
        "MAE": mean_absolute_error(y_eval, predictions),
        "MSE": mean_squared_error(y_eval, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_eval, predictions)),
        "R²": r2_score(y_eval, predictions),
    }

## 9. Train a Linear Regression Model

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_results = evaluate_regression(linear_model, X_test, y_test)
pd.Series(linear_results, name="Linear Regression").round(3)

In [ ]:
linear_predictions = linear_model.predict(X_test)

linear_comparison = pd.DataFrame({
    "Actual Sales": y_test.to_numpy(),
    "Predicted Sales": linear_predictions
})
linear_comparison.head(10).round(2)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, linear_predictions, alpha=0.7)
limits = [
    min(y_test.min(), linear_predictions.min()),
    max(y_test.max(), linear_predictions.max())
]
plt.plot(limits, limits, "r--")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

## 10. Train a Decision Tree Regressor

A Decision Tree learns a series of feature-based splits. An unrestricted tree can fit the training data extremely closely, which may lead to overfitting.

In [ ]:
decision_tree = DecisionTreeRegressor(random_state=42)
decision_tree.fit(X_train, y_train)

dt_predictions = decision_tree.predict(X_test)
dt_results = evaluate_regression(decision_tree, X_test, y_test)

print("Training R²:", round(decision_tree.score(X_train, y_train), 3))
print("Testing R²:", round(dt_results["R²"], 3))
pd.Series(dt_results, name="Decision Tree").round(3)

In [ ]:
dt_comparison = pd.DataFrame({
    "Actual Sales": y_test.to_numpy(),
    "Predicted Sales": dt_predictions
})
dt_comparison.head(10).round(2)

## 11. Tune Decision Tree Hyperparameters

We explore `max_depth` and `min_samples_split`.

- `max_depth` limits how deep the tree can grow.
- `min_samples_split` sets the minimum number of samples required to split a node.

**Caution:** The earlier exploration compared test-set scores across settings, so the test set influenced the tuning decision. For a more rigorous estimate, use cross-validation for model selection and keep a final test set untouched.

In [ ]:
depth_results = []

for depth in [2, 3, 4, 5, 6, 8, 10]:
    candidate = DecisionTreeRegressor(max_depth=depth, random_state=42)
    candidate.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth,
        "Training R²": candidate.score(X_train, y_train),
        "Testing R²": candidate.score(X_test, y_test),
    })

depth_results_df = pd.DataFrame(depth_results)
depth_results_df.round(3)

In [ ]:
split_results = []

for min_split in [2, 5, 10, 20, 30]:
    candidate = DecisionTreeRegressor(
        max_depth=10,
        min_samples_split=min_split,
        random_state=42
    )
    candidate.fit(X_train, y_train)
    split_results.append({
        "min_samples_split": min_split,
        "Training R²": candidate.score(X_train, y_train),
        "Testing R²": candidate.score(X_test, y_test),
    })

split_results_df = pd.DataFrame(split_results)
split_results_df.round(3)

In [ ]:
tuned_tree = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=2,
    random_state=42
)
tuned_tree.fit(X_train, y_train)

tuned_tree_results = evaluate_regression(tuned_tree, X_test, y_test)
print("Training R²:", round(tuned_tree.score(X_train, y_train), 3))
pd.Series(tuned_tree_results, name="Tuned Decision Tree").round(3)

## 12. Train a Random Forest Regressor

A Random Forest combines predictions from many decision trees. In the previous notebook run, the saved Random Forest pipeline achieved MAE ≈ 0.620, RMSE ≈ 0.769, and R² ≈ 0.981 on the same test split, outperforming the tuned Decision Tree's recorded test metrics. Re-run the cell below to reproduce the comparison in this notebook.

In [ ]:
rf_pipeline = Pipeline([
    ("model", RandomForestRegressor(
        max_depth=10,
        min_samples_split=2,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_results = evaluate_regression(rf_pipeline, X_test, y_test)
pd.Series(rf_results, name="Random Forest").round(3)

## 13. Compare Models

The table summarizes test-set metrics. The best choice should consider both error metrics and generalization, not just one score.

In [ ]:
results_table = pd.DataFrame([
    {"Model": "Linear Regression", **linear_results},
    {"Model": "Decision Tree", **dt_results},
    {"Model": "Tuned Decision Tree", **tuned_tree_results},
    {"Model": "Random Forest", **rf_results},
]).set_index("Model")

results_table.round(3)

In [ ]:
results_table["R²"].sort_values().plot(kind="barh", figsize=(8, 4))
plt.xlabel("Test R²")
plt.title("Test-set R² Comparison")
plt.tight_layout()
plt.show()

## 14. Select and Fit the Final Pipeline

Based on the recorded results, Random Forest is the current preferred candidate because it had lower prediction errors and higher test R² than the tuned Decision Tree. This selection should ideally be confirmed with cross-validation before production use.

A pipeline packages the model as one reusable object. This pipeline currently contains the estimator only; no scaling step is needed for a Random Forest.

In [ ]:
final_pipeline = Pipeline([
    ("model", RandomForestRegressor(
        max_depth=10,
        min_samples_split=2,
        random_state=42
    ))
])

final_pipeline.fit(X_train, y_train)
print(final_pipeline)

In [ ]:
final_predictions = final_pipeline.predict(X_test)
final_results = evaluate_regression(final_pipeline, X_test, y_test)

print("Training R²:", round(final_pipeline.score(X_train, y_train), 4))
print("Testing R²:", round(final_results["R²"], 4))
pd.Series(final_results, name="Final Random Forest").round(4)

### Interpretation

The final test metrics estimate how this fitted model performs on the held-out test subset. A strong test score is encouraging, but a single split can vary. Cross-validation can provide a more robust estimate. Avoid using the test set repeatedly to choose hyperparameters, because that can make the reported score optimistic.

## 15. Save the Final Pipeline

Save the trained pipeline with Joblib so another Python script or a FastAPI application can load it without retraining.

In [ ]:
MODEL_PATH = "sales_prediction_pipeline.pkl"
joblib.dump(final_pipeline, MODEL_PATH)
print(f"Final pipeline saved to: {os.path.abspath(MODEL_PATH)}")

## 16. Reload and Verify the Saved Model

Load the saved file and confirm the estimator and expected feature names.

In [ ]:
loaded_model = joblib.load(MODEL_PATH)
print(loaded_model)
print("Expected features:", loaded_model.feature_names_in_)

## 17. Predict Sales for New Advertising Data

Provide one row with the same feature names and feature order used during training. The values should be in the same units as the training dataset.

In [ ]:
new_data = pd.DataFrame({
    "TV": [150.0],
    "radio": [25.0],
    "newspaper": [20.0]
})

predicted_sales = loaded_model.predict(new_data)
print("Predicted Sales:", round(float(predicted_sales[0]), 2))

## 18. Conclusion

This project explored sales prediction using TV, radio, and newspaper advertising expenditure. The workflow included data inspection, exploratory analysis, feature/target selection, a reproducible train-test split, regression model training, evaluation, Decision Tree tuning, and Random Forest comparison.

The recorded Random Forest test results were stronger than the tuned Decision Tree results. The final model is saved as `sales_prediction_pipeline.pkl` and can be loaded for predictions or integrated into a FastAPI application.

### Future Improvements
- Use cross-validation for model selection and hyperparameter tuning.
- Keep a final test set untouched until model selection is complete.
- Add input validation and error handling to the API.
- Document environment setup and usage in a GitHub README.